# Firmware benchmark analysis template

**No hardware measurements are included.** The default CSV has a header only. This notebook compares equivalent polling, interrupt, and DMA acquisition runs. Read [the measurement contract](README.md) first.

Optional synthetic mode exists only to demonstrate analysis. Its numbers have no device-performance meaning. Do not interpret acquisition elapsed time as CPU utilization.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works when launched from the repository root or the benchmarks folder.
HERE = Path.cwd()
BENCH = HERE / "benchmarks" if (HERE / "benchmarks").is_dir() else HERE
CSV_PATH = BENCH / "data" / "measurements-template.csv"
USE_SYNTHETIC = False
METHODS = ["polling", "interrupt", "dma"]
LABEL = "SYNTHETIC DEMONSTRATION — NOT HARDWARE RESULTS" if USE_SYNTHETIC else "Imported measurements — verify provenance"


In [ ]:
if USE_SYNTHETIC:
    # Deliberately similar acquisition durations: CPU load is NOT represented.
    rng = np.random.default_rng(51)
    rows = []
    for method in METHODS:
        for run in range(1, 6):
            rows.append({"method": method, "run": str(run),
                         "elapsed_us": float(rng.normal(100000, 300)),
                         "sample_count": 1000, "bytes_per_sample": 2,
                         "target_interval_us": 100.0,
                         "jitter_us": float(rng.uniform(0.5, 2.5))})
    data = pd.DataFrame(rows)
else:
    data = pd.read_csv(CSV_PATH, dtype={"method": "string", "run": "string"})
print(LABEL)
print(f"Input rows: {len(data)}")


In [ ]:
required = ["method", "run", "elapsed_us", "sample_count", "bytes_per_sample", "target_interval_us", "jitter_us"]
missing = set(required) - set(data.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if not data.empty:
    if data[required].isna().any().any():
        raise ValueError("Required fields must not contain missing values.")
    unknown = set(data["method"]) - set(METHODS)
    if unknown:
        raise ValueError(f"Unknown methods: {unknown}")
    if data["run"].astype(str).str.strip().eq("").any():
        raise ValueError("Run IDs must be non-empty.")
    if data.duplicated(["method", "run"]).any():
        raise ValueError("Run IDs must be unique within each method.")
    numeric = required[2:]
    data[numeric] = data[numeric].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(data[numeric].to_numpy(dtype=float)).all():
        raise ValueError("Numeric fields must be finite.")
    if (data[["elapsed_us", "bytes_per_sample", "target_interval_us"]] <= 0).any().any():
        raise ValueError("Time, sample width, and target interval must be positive.")
    if (data["sample_count"] < 2).any() or (data["jitter_us"] < 0).any():
        raise ValueError("Need >= 2 samples and non-negative jitter.")
    for field in ["sample_count", "bytes_per_sample"]:
        if (data[field] % 1 != 0).any():
            raise ValueError(f"{field} must be an integer.")
    for field in ["sample_count", "bytes_per_sample", "target_interval_us"]:
        if data[field].nunique() != 1:
            raise ValueError(f"Unequal {field}: compare equivalent workloads separately.")
    data["samples_per_s"] = data["sample_count"] * 1e6 / data["elapsed_us"]
    data["bytes_per_s"] = data["samples_per_s"] * data["bytes_per_sample"]
    absent = set(METHODS) - set(data["method"])
    if absent:
        print(f"Partial comparison; missing methods: {sorted(absent)}")
    display(data)
else:
    print("No measurements. Supply a real CSV or explicitly enable synthetic demonstration mode.")


## Compare repeated runs

`elapsed_std_us` is **between-run** variation. `mean_jitter_us` is the average **within-run standard deviation of inter-sample intervals**, as supplied by the measurement process. These are different quantities. One run has undefined between-run standard deviation; a zero-length error bar below then means unavailable, not zero variation.


In [ ]:
summary = pd.DataFrame()
if not data.empty:
    summary = data.groupby("method").agg(
        runs=("run", "count"), mean_elapsed_us=("elapsed_us", "mean"),
        elapsed_std_us=("elapsed_us", "std"),
        mean_samples_per_s=("samples_per_s", "mean"),
        mean_bytes_per_s=("bytes_per_s", "mean"),
        mean_jitter_us=("jitter_us", "mean"),
    ).reindex([m for m in METHODS if m in set(data["method"])])
    print(LABEL)
    display(summary)
    if (summary["runs"] < 2).any():
        print("At least one method has < 2 runs: between-run variation is unavailable.")
else:
    print("Summary pending real measurements.")


In [ ]:
if not summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    colors = ["#607d8b", "#00897b", "#5c6bc0"][:len(summary)]
    axes[0].bar(summary.index, summary["mean_elapsed_us"],
                yerr=summary["elapsed_std_us"].fillna(0), color=colors, capsize=4)
    axes[0].set(title="Acquisition elapsed time", ylabel="Microseconds (mean ± run SD)")
    axes[1].bar(summary.index, summary["mean_bytes_per_s"] / 1e6, color=colors)
    axes[1].set(title="Effective data throughput", ylabel="MB/s (decimal)")
    axes[2].bar(summary.index, summary["mean_jitter_us"], color=colors)
    axes[2].set(title="Inter-sample interval jitter", ylabel="Microseconds (mean within-run SD)")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
    fig.suptitle(LABEL, fontsize=11)
    plt.show()
else:
    print("Plots pending measurements; no performance values were invented.")


## Interpretation and run record

Record the source commit, tool/core/compiler versions, clocks, instrument, timing boundaries, repeat policy, input conditions, overflow checks, and raw timestamp files. Describe uncertainty and any excluded runs with reasons.

- Which methods produced equivalent, correct data?
- Were the timing differences larger than measurement resolution and overhead?
- Was elapsed time constrained by the peripheral sampling rate?
- What separate evidence would be needed to claim a reduction in CPU service time?

**Conclusions:** Pending real hardware measurements. Clear all outputs before committing the template.
